In [4]:
import tkinter as tk
from tkinter import messagebox
from tkinter import ttk
import joblib
import csv
import os
from datetime import datetime


# Load the saved model, TF-IDF vectorizer and label encoder.
try:

    best_xgb_model = joblib.load(
        "cyber_xgb_model.pkl"
    )

    tfidf = joblib.load(
        "cyber_tfidf.pkl"
    )

    label_encoder = joblib.load(
        "cyber_label_encoder.pkl"
    )

    print("Model loaded successfully.")
    print("Model classes:", best_xgb_model.classes_)
    print("Label Encoder classes:", label_encoder.classes_)

    if len(best_xgb_model.classes_) != len(label_encoder.classes_):

        raise ValueError(
            "The XGBoost model and Label Encoder "
            "have different numbers of classes."
        )

except Exception as e:

    messagebox.showerror(
        "Loading Error",
        "The required model files could not be loaded.\n\n"
        + str(e)
    )

    raise SystemExit


# Create the main application window.
window = tk.Tk()

window.title(
    "Cyberbullying Detection Prototype"
)

window.geometry(
    "1150x700"
)

window.configure(
    bg="#F4F6F8"
)

window.resizable(
    True,
    True
)


# Create the main heading.
heading = tk.Label(

    window,

    text="Cyberbullying Detection Prototype",

    font=("Arial", 22, "bold"),

    bg="#1F4E79",

    fg="white",

    pady=15

)

heading.pack(
    fill="x"
)


# Create a subtitle.
subtitle = tk.Label(

    window,

    text="Multi-Class Social Media Comment Classification",

    font=("Arial", 11),

    bg="#D9EAF7",

    fg="#1F2937",

    pady=8

)

subtitle.pack(
    fill="x"
)


# Create the main content area.
main_frame = tk.Frame(

    window,

    bg="#F4F6F8"

)

main_frame.pack(
    fill="both",
    expand=True,
    padx=20,
    pady=20
)


# Create the prediction panel on the left.
prediction_frame = tk.LabelFrame(

    main_frame,

    text="  Cyberbullying Prediction  ",

    font=("Arial", 14, "bold"),

    bg="white",

    fg="#1F4E79",

    bd=2,

    relief="groove",

    padx=15,

    pady=15

)

prediction_frame.pack(

    side="left",

    fill="both",

    expand=True,

    padx=(0, 10)

)


# Create the instruction.
instruction = tk.Label(

    prediction_frame,

    text="Enter a social media comment below:",

    font=("Arial", 12, "bold"),

    bg="white",

    fg="#333333"

)

instruction.pack(

    anchor="w",

    pady=(5, 8)

)


# Create a frame for the text box and scrollbar.
text_frame = tk.Frame(

    prediction_frame,

    bg="white"

)

text_frame.pack(

    fill="both",

    expand=False

)


# Create the comment text box.
text_box = tk.Text(

    text_frame,

    height=12,

    width=55,

    font=("Arial", 11),

    wrap=tk.WORD,

    bg="#FAFAFA",

    fg="#222222",

    insertbackground="#1F4E79",

    relief="solid",

    bd=1,

    padx=8,

    pady=8

)

text_box.pack(

    side="left",

    fill="both",

    expand=True

)


# Create the scrollbar for the comment box.
text_scrollbar = tk.Scrollbar(

    text_frame,

    orient="vertical",

    command=text_box.yview

)

text_scrollbar.pack(

    side="right",

    fill="y"

)


text_box.configure(

    yscrollcommand=text_scrollbar.set

)


# Store the current prediction.
current_prediction = tk.StringVar()

current_prediction.set("")


# Create the prediction result area.
result_frame = tk.Frame(

    prediction_frame,

    bg="#EAF2F8",

    bd=1,

    relief="solid"

)

result_frame.pack(

    fill="x",

    pady=18

)


result_label = tk.Label(

    result_frame,

    text="Prediction will appear here",

    font=("Arial", 16, "bold"),

    bg="#EAF2F8",

    fg="#1F4E79",

    pady=15

)

result_label.pack(
    fill="x"
)


# Create the prediction button.
predict_button = tk.Button(

    prediction_frame,

    text="Predict",

    font=("Arial", 12, "bold"),

    width=16,

    bg="#1F4E79",

    fg="white",

    activebackground="#163A5C",

    activeforeground="white",

    cursor="hand2",

    bd=0,

    padx=10,

    pady=8,

    command=lambda: predict_text()

)

predict_button.pack(
    pady=5
)


# Create the clear button.
clear_button = tk.Button(

    prediction_frame,

    text="Clear",

    font=("Arial", 11, "bold"),

    width=16,

    bg="#D9E2F3",

    fg="#1F2937",

    activebackground="#B4C7E7",

    cursor="hand2",

    bd=0,

    padx=10,

    pady=7,

    command=lambda: clear_text()

)

clear_button.pack(
    pady=5
)


# Create the human evaluation panel on the right.
evaluation_frame = tk.LabelFrame(

    main_frame,

    text="  Human Evaluation  ",

    font=("Arial", 14, "bold"),

    bg="white",

    fg="#548235",

    bd=2,

    relief="groove",

    padx=15,

    pady=15

)

evaluation_frame.pack(

    side="right",

    fill="both",

    expand=True,

    padx=(10, 0)

)


# Create evaluation instructions.
evaluation_instruction = tk.Label(

    evaluation_frame,

    text=(
        "Please rate the prototype from 1 to 5.\n"
        "1 = Very Poor    5 = Excellent"
    ),

    font=("Arial", 10),

    bg="white",

    fg="#444444",

    justify="left"

)

evaluation_instruction.pack(

    anchor="w",

    pady=(5, 15)

)


# Create rating variables.
rating1 = tk.StringVar(value="Select")
rating2 = tk.StringVar(value="Select")
rating3 = tk.StringVar(value="Select")
rating4 = tk.StringVar(value="Select")
rating5 = tk.StringVar(value="Select")


rating_values = [
    "Select",
    "1",
    "2",
    "3",
    "4",
    "5"
]


# Function for creating rating rows.
def create_rating_row(parent, text, variable):

    row = tk.Frame(

        parent,

        bg="white"

    )

    row.pack(

        fill="x",

        pady=6

    )

    label = tk.Label(

        row,

        text=text,

        font=("Arial", 10, "bold"),

        bg="white",

        fg="#333333",

        width=25,

        anchor="w"

    )

    label.pack(
        side="left"
    )

    dropdown = ttk.Combobox(

        row,

        textvariable=variable,

        values=rating_values,

        state="readonly",

        width=12

    )

    dropdown.pack(
        side="right"
    )

    dropdown.current(0)


# Create the five evaluation factors.
create_rating_row(
    evaluation_frame,
    "1. Ease of Use",
    rating1
)

create_rating_row(
    evaluation_frame,
    "2. Prediction Clarity",
    rating2
)

create_rating_row(
    evaluation_frame,
    "3. Prediction Relevance",
    rating3
)

create_rating_row(
    evaluation_frame,
    "4. Interface Design",
    rating4
)

create_rating_row(
    evaluation_frame,
    "5. Overall Satisfaction",
    rating5
)


# Create feedback heading.
feedback_label = tk.Label(

    evaluation_frame,

    text="Additional Feedback",

    font=("Arial", 11, "bold"),

    bg="white",

    fg="#548235"

)

feedback_label.pack(

    anchor="w",

    pady=(18, 6)

)


# Create feedback frame.
feedback_frame = tk.Frame(

    evaluation_frame,

    bg="white"

)

feedback_frame.pack(

    fill="both",

    expand=True

)


# Create feedback text box.
feedback_box = tk.Text(

    feedback_frame,

    height=8,

    width=45,

    font=("Arial", 10),

    wrap=tk.WORD,

    bg="#FAFAFA",

    fg="#222222",

    insertbackground="#548235",

    relief="solid",

    bd=1,

    padx=8,

    pady=8

)

feedback_box.pack(

    side="left",

    fill="both",

    expand=True

)


# Create scrollbar for feedback.
feedback_scrollbar = tk.Scrollbar(

    feedback_frame,

    orient="vertical",

    command=feedback_box.yview

)

feedback_scrollbar.pack(

    side="right",

    fill="y"

)


feedback_box.configure(

    yscrollcommand=feedback_scrollbar.set

)


# Create the submit evaluation button.
submit_button = tk.Button(

    evaluation_frame,

    text="Submit Evaluation",

    font=("Arial", 12, "bold"),

    bg="#548235",

    fg="white",

    activebackground="#3F6228",

    activeforeground="white",

    cursor="hand2",

    bd=0,

    padx=15,

    pady=9,

    command=lambda: submit_evaluation()

)

submit_button.pack(

    pady=15

)


# Create the prediction function.
def predict_text():

    try:

        user_text = text_box.get(
            "1.0",
            tk.END
        ).strip()

        if len(user_text) == 0:

            messagebox.showwarning(

                "Input Required",

                "Please enter a social media comment."

            )

            return


        transformed_text = tfidf.transform(
            [user_text]
        )


        prediction = best_xgb_model.predict(
            transformed_text
        )


        predicted_id = int(
            prediction[0]
        )


        if predicted_id < 0 or predicted_id >= len(
            label_encoder.classes_
        ):

            raise ValueError(
                "Invalid predicted class."
            )


        predicted_class = label_encoder.inverse_transform(
            [predicted_id]
        )[0]


        current_prediction.set(
            str(predicted_class)
        )


        result_label.config(

            text=(
                "Predicted Class: "
                + str(predicted_class)
            ),

            fg="#1F4E79",

            bg="#EAF2F8"

        )


        
        print("PREDICTION RESULT")
        print("Input:", user_text)
        print("Predicted Class ID:", predicted_id)
        print("Predicted Class:", predicted_class)


    except Exception as e:

        messagebox.showerror(

            "Prediction Error",

            "An error occurred while making "
            "the prediction:\n\n"
            + str(e)

        )


# Create the clear function.
def clear_text():

    text_box.delete(
        "1.0",
        tk.END
    )

    result_label.config(

        text="Prediction will appear here",

        fg="#1F4E79",

        bg="#EAF2F8"

    )

    current_prediction.set("")

    rating1.set("Select")
    rating2.set("Select")
    rating3.set("Select")
    rating4.set("Select")
    rating5.set("Select")

    feedback_box.delete(
        "1.0",
        tk.END
    )


# Create the human evaluation submission function.
def submit_evaluation():

    try:

        if current_prediction.get() == "":

            messagebox.showwarning(

                "Prediction Required",

                "Please enter a comment and generate "
                "a prediction before submitting the evaluation."

            )

            return


        ratings = [

            rating1.get(),
            rating2.get(),
            rating3.get(),
            rating4.get(),
            rating5.get()

        ]


        if "Select" in ratings:

            messagebox.showwarning(

                "Ratings Required",

                "Please provide a rating from 1 to 5 "
                "for all five evaluation factors."

            )

            return


        feedback = feedback_box.get(

            "1.0",

            tk.END

        ).strip()


        if len(feedback) == 0:

            messagebox.showwarning(

                "Feedback Required",

                "Please provide written feedback "
                "before submitting the evaluation."

            )

            return


        timestamp = datetime.now().strftime(

            "%Y-%m-%d %H:%M:%S"

        )


        filename = (
            "cyber_human_evaluation.csv"
        )


        file_exists = os.path.isfile(
            filename
        )


        with open(

            filename,

            "a",

            newline="",

            encoding="utf-8"

        ) as file:

            writer = csv.writer(
                file
            )


            if not file_exists:

                writer.writerow([

                    "Timestamp",

                    "Predicted Class",

                    "Ease of Use",

                    "Prediction Clarity",

                    "Prediction Relevance",

                    "Interface Design",

                    "Overall Satisfaction",

                    "Feedback"

                ])


            writer.writerow([

                timestamp,

                current_prediction.get(),

                rating1.get(),

                rating2.get(),

                rating3.get(),

                rating4.get(),

                rating5.get(),

                feedback

            ])


        messagebox.showinfo(

            "Evaluation Submitted",

            "Thank you. Your ratings and feedback "
            "have been recorded successfully."

        )


        rating1.set("Select")
        rating2.set("Select")
        rating3.set("Select")
        rating4.set("Select")
        rating5.set("Select")

        feedback_box.delete(
            "1.0",
            tk.END
        )


    except Exception as e:

        messagebox.showerror(

            "Evaluation Error",

            "The evaluation could not be saved:\n\n"
            + str(e)

        )


# Start the application.
window.mainloop()

Model loaded successfully.
Model classes: [0 1 2 3 4 5]
Label Encoder classes: ['age' 'ethnicity' 'gender' 'not_cyberbullying' 'other_cyberbullying'
 'religion']
